# Encoding Geographical Risk from a Global Sanction Datasets

## A Structured Open Source Intelligence Tool to support Regulatory and AML/CFT Compliance
---

**Author**: Francesco Silvano, MEng in Engineering Management  

**Instructors**: Prof. Marco Maggini, Prof. Federico Becattini

**Course**: Business Intelligence, University of Siena 

**Publication Date**: June 2026

---

## Introduction & Context

One of the biggest challenges in AML/CFT is to let different databases to communicate efficiently. Analysts require powerful tools to handle complex data collection, analysis and decision-making. Not everytime it's possible to do it efficiently and cross-referencing datasets is still very time-consuming and costly process. Businesses engage with dozens of counterparties every day and face structural information asymmetry with measurable consequences for risk. 

Many open-source solutions exist, but tend to be inferior in terms of data availability, quality, retrieval speed and user-friendliness, particularly when it comes to high-stakes sanctions compliance involving intricate data (e.g. verifications for financial transactions).


This project therefore proposes a sanctions intelligence system that uses open-source datasets to map entity relationships, detect evasion patterns and provide actionable insights via an interactive dashboard. It identifies potential patterns and factors influencing sanctions risks, thereby informing better compliance practices and policy decisions within the industry. 

The dataset includes a comprehensive collection of sanctions-related entities and connections. The analysis focuses on identifying evasion patterns and risk propagation.

---

### Methodology

The datasets employed in this analysis are publicly sourced from [OpenSanctions](https://opensanctions.org), a public repository of global sanction data. The datasets include information on sanctioned entities - people, organizations and vessels - as well as complementary data on politically exposed persons (PEPs) and financial securities. 

The built pipeline ingests and combines multiple datasets, clean data ensuring that datasets are in a usable format and that any inconsistencies or missing values are addressed; an enrichment layer includes sanctioned maritime vessels and financial securities through complementary data joins - providing an holistic view of the activities carried out by sanctioned entities.

- In _Part I_ an exploration of the datasets is perfomed to understand the distribution and quality of data across different types of entities and countries;
- in _Part II_ we build the analytical layers: the data mining layer attempts to identify frequent patterns and associations in the data using an **A-Priori algorithm** for frequent itemsets and **Jaccard Similarity** for entity deduplication and collaborative filtering, while a recommendation system based on UV-decomposition via **Stochastic Gradient Descent (SGD)** suggests potential connections between entities based on their attributes and relationships; 
- In _Part III_, a **simple machine learning model** is proposed to predict risk levels based on geographical and entity-specific features, providing a probabilistic assessment of potential sanctions evasion.

#### A few assumptions on data quality to be aware of

- The analysis is based on the assumption that the datasets used are comprehensive and accurate representations of the entities and relationships involved in sanctions. While OpenSanctions is an authorative source, and their updates are frequent, the available data may not capture all information in real time, and there may be gaps or inaccuracies.

- Data is distributed in the **_FollowTheMoney (FTM)_** format, which is a specific structure for representing entities and their relationships. This format is in JSON, which requires significant preprocessing to be transformed into a structured format suitable for analysis. For the purpose of this project, a simpler version of the data in CSV format is used, which may not capture all the nuances and details present in the original JSON data, such as the metadata on the source and confidence level of each entry. A pratical example: sanctions designation wouldn’t be in the PEPs collection because those sources don’t contain the designations texts. 

- The data originates from a variety of sources and therefore varies in terms of detail and precision (e.g., many listings of political office holders do not include their exact birth dates or full given names). This represents a significant challenge for data quality, as it can lead to difficulties in entity resolution and risk assessment. For instance, the lack of precise identifiers may result in false positives or negatives when screening for sanctioned entities, which can have serious implications for compliance efforts.

- The cut-off date for the data is set to 10/06/2026, and any events or changes that occurred after this date are not included in the analysis. This means that the findings may not reflect the most current state of sanctions and related activities.

---

### Technology Dependencies

```
0 Setup
└── 1 Ingest
    └── 2 Features
        ├── 3 Jaccard Dedup  ──────────────────────────────┐
        │   └── [deduped]                                  │
        │       ├── 4 Frequent Itemsets                    │
        │       │   └── [rules, fi_flags]                  │
        │       ├── 5 Network                              │
        │       │   └── [G, centrality, communities]       │
        │       │       ├── 6 Recommendation System        │
        │       │       │   └── [risk_score via UV]        │
        │       │       └── 7 ML Classifier ───────────────┘
        │       │           └── [risk_score]
        │       └── 8 Evasion Detection
        │           └── [evasion_score, evasion_flag]
        └── 9 Quarto Dashboard for Visualization
```

### Possible applications
---

| # | Application | Description | Example |
| --- | --- | --- | --- |
| 1 | AML/CFT Counterparty Screening | Automated screening of counterparties, vendors, and transaction parties against the consolidated sanctions list. Fuzzy matching catches name variants and transliterations used to evade direct string matches — a common evasion technique. | A correspondent bank screens incoming wire transfers; the fuzzy matching layer flags "Rosneft Trading S.A." as a near-match to the designated "Rosneft Trading SA" on the EU FSF list, triggering an enhanced due diligence review before funds are released. |
| 2 | Sanctions Evasion Pattern Detection | Frequent itemset mining surfaces co-occurrence patterns in entity attributes — jurisdiction, schema type, cross-list membership — that characterise known evasion structures such as offshore shells, conduit flag states, and layered ownership chains. | The A-Priori algorithm identifies that entities sharing the profile {Company, country:AE, listed in OFAC} disproportionately lack an EU FSF designation. This cross-list coverage gap is consistent with UAE-incorporated intermediaries used to route sanctioned capital between Russian principals and European markets. |
| 3 | Geopolitical and Sectoral Risk Intelligence | Network centrality analysis maps which industries, jurisdictions, and corporate archetypes concentrate the most sanctioned connections — exposing emerging evasion corridors before they appear in formal designations. | A defense ministry analyst uses betweenness centrality scores to identify a cluster of Turkish shipping companies acting as structural brokers between sanctioned Russian oil producers and Asian buyers — a pattern not yet reflected in any active designation list. |
| 4 | Investigative Research and OSINT | An interactive dashboard for analysts, journalists, and NGOs to explore entity networks, trace ownership chains, and surface previously unknown connections between designated persons and ostensibly clean entities. The recommendation system surfaces candidate connections the user has not yet examined. | An investigative journalist queries the dashboard to trace the ownership chain of a vessel flagged under a conduit jurisdiction, linking it through three shell companies back to a sanctioned oligarch whose name does not appear in any public vessel registry. |

---

### Libraries and Imports

In [ ]:
# Data
import pandas as pd
import numpy as np
from scipy import stats

# Other tools
import json, re, hashlib, itertools, collections, math, warnings
from pathlib import Path

# Similarity
from datasketch import MinHash, MinHashLSH        # MinHash-based Jaccard LSH

# A-Priori Algorithm
from mlxtend.frequent_patterns import apriori, association_rules 
from mlxtend.preprocessing import TransactionEncoder

# LDA, QDA use module: scikit learn
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.inspection import DecisionBoundaryDisplay

# Vizualization
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

# Machine Learning
# from sklearn.linear_model import SGDClassifier
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import (classification_report, roc_auc_score,
#                              average_precision_score)
# from sklearn.model_selection import StratifiedKFold
# from scipy.sparse import lil_matrix, csr_matrix

DATA_PATH = "data/input"

### Collection and Update Process

The dataset coverage — i.e. when it was started and how frequently it is updated — is every six hours, and reflects the latest submissions and updates from the original sources. 

The dataset source is updated automatically; however, for the purposes of this project, a static snapshot of the data was taken on 10 June 2026, the date on which the final analysis was run, to ensure consistency across the analysis and avoid potential issues arising from changes to the data during the project timeline.

## Part I: Data Exploration and Preprocessing
---

### Cleaning the main datasets

Although no extensive cleaning process was required for the datasets, as they were already in a structured format and did not contain significant missing values or inconsistencies, some preprocessing steps were necessary to ensure the data was in a usable format for analysis.

The PEPs dataset, in particular, required a deduplication process due to the presence of duplicates across different sources.

The deduplication process was implemented by simply keeping the occurrence with the most complete information, i.e. the row with the least number of missing values across all columns. This approach ensured that we retained the most informative record for each unique entity while eliminating redundant entries that could jeopardize the accuracy of our analysis. 

### Describing the main datasets

#### DS1 Consolidated Sanctions

The _"Consolidated Sanctions"_ dataset is a distribution listing all the sanctioned entities designated by different countries and  international organisations. This can include military, trade and travel restrictions.

This collection combines all designated entities from the various sanctions lists included in OpenSanctions. It represents the broadest set of sanctioned people, companies and other assets that is designated globally.

In [ ]:
# Load the data for Consolidated Sanctions dataset
df_consolidated = pd.read_csv(f"{DATA_PATH}/consolidatedsanctions/targets.simple-1.csv", low_memory=False).drop(columns=['phones','emails']) # phones and emails are dropped for simplicity
df_consolidated['countries'] = df_consolidated['countries'].str.upper()

Some records may have multiple `countries` separated by a semicolon. This means that specific entry has links to two countries. For example, a string like "RU;CY" could have different meanings such as _"Russian oligarch with Cypriot residency"_ or _"Russian company with a Cypriot branch"_.

In [ ]:
df_consolidated['countries'].str.contains(';', na=False).sum()

The number of those entities is **4575**. If left as-is, each entity would appear zero times in the RU count and zero times in the CY count. It almost certainly falls below a 2% threshold as a combined string, so it lands in the "Others" category, so this category would be artificially inflated. This represents a systematic bias against multi-jurisdiction entities. 

Therefore multi-country entities are disproportionately the analytically interesting ones - dual-national oligarchs, shell companies registered across multiple jurisdictions, intermediaries with deliberate geographic complexity.

The correct interpretation, the one used throughout the project is _**"How many sanctioned entities have a nexus to each country?"**_

#### DS2 Politically Exposed Persons (PEPs) dataset

The _"Politically Exposed Persons (PEPs)"_ dataset is a collection of individuals who are or have been entrusted with prominent public roles.

This dataset is important for financial institutions and other entities that are required to conduct enhanced due diligence on PEPs to mitigate risks of corruption and money laundering. It includes information on the individual's name, position, country of origin, and any associated risks.

In [ ]:
# Load the data for the PEPs dataset
df_peps = pd.read_csv(f'{DATA_PATH}/peps/targets.simple-1.csv', low_memory=False) # low_memory=False to avoid dtype warnings
df_peps['countries'] = df_peps['countries'].str.upper()
df_peps

In [ ]:
df_peps.head()
n_peps = df_peps['id'].nunique()
n_duplicates = df_peps['name'].duplicated().sum()

print(f"Number of unique entities in PEPs dataset: {n_peps}")
print(f"Number of duplicates in PEPs dataset: {n_duplicates}")

df_duplicates = df_peps[df_peps['name'].duplicated(keep=False)].sort_values('name')
df_duplicates

In [ ]:
df_peps_work = df_peps.copy()
df_peps_work['non_null_count'] = df_peps_work.count(axis=1)

df_peps_work = df_peps_work.sort_values('non_null_count', ascending=False).drop_duplicates(subset='name', keep='first')

In [ ]:
df_peps_work

In [ ]:
df_sanctioned_people = df_consolidated[df_consolidated['schema'] == 'Person']
df_sanctioned_people

The `df_sanctioned_people` dataframe contains only the entities from the Consolidated Sanctions dataset that are classified as 'Person' under the 'schema' column. These are all the individuals that have been designated as sanctioned entities by different countries and international organisations.

#### Distribution of sanctioned individuals by country

##### Sanctioned individuals that are not PEPs

In [ ]:
df_sanctioned_people_nopeps = df_sanctioned_people[~df_sanctioned_people['id'].isin(df_peps_work['id'])]
df_sanctioned_people_nopeps

In [ ]:
country_counts_nopeps = df_sanctioned_people_nopeps['countries'].str.split(';').explode().str.strip().value_counts()

country_percentage_nopeps = country_counts_nopeps / country_counts_nopeps.sum()

main_countries_nopeps = country_counts_nopeps[country_percentage_nopeps >= 0.02]
other_countries_nopeps = country_counts_nopeps[country_percentage_nopeps < 0.02].sum()

plot_data = main_countries_nopeps.copy()
if other_countries_nopeps > 0:
    plot_data['Others'] = other_countries_nopeps

fig = plt.figure(figsize=(10, 6))
plot_data.plot(
    kind='pie',
    autopct='%1.1f%%', # autopct to show percentage on the pie chart
    startangle=90,
    pctdistance=0.8
)

plt.title('Distribution of sanctioned individuals by country')
plt.axis('equal')
plt.tight_layout()
plt.legend()
plt.show()

##### Sanctioned individuals that are also PEPs

Analyzing the distribution of the sanctioned PEPs by country allows us to understand how many politically exposed persons are sanctioned in each country, and to identify any patterns or trends in the distribution of sanctions across different countries. This can provide insights into the geopolitical and economic factors that may influence sanctions designation, and can inform risk assessment and compliance efforts for entities operating in those countries.

In [ ]:
df_sanctioned_people_peps = pd.merge(df_sanctioned_people, df_peps_work, on='id', how='inner', suffixes=('_consolidated', '_peps'))
df_sanctioned_people_peps

The resulting dataset is typically named "The Russian Government" by the analysts for the presence of many high-profile Russian politicians and oligarchs who are both sanctioned and politically exposed. This subset can be analyzed further to understand the overlap between the two datasets and to identify any unique characteristics of these entities.

In [ ]:
country_counts_peps = df_sanctioned_people_peps['countries_peps'].str.split(';').explode().str.strip().value_counts()

country_percentage_peps = country_counts_peps / country_counts_peps.sum()

main_countries_peps = country_counts_peps[country_percentage_peps >= 0.02]
other_countries_peps = country_counts_peps[country_percentage_peps < 0.02].sum()

plot_data = main_countries_peps.copy()
if other_countries_peps > 0:
    plot_data['Others'] = other_countries_peps

fig = plt.figure(figsize=(10, 6))
plot_data.plot(
    kind='pie',
    autopct='%1.1f%%', # autopct to show percentage on the pie chart
    startangle=90,
    pctdistance=0.8
)

plt.title('Distribution of sanctioned PEPs by country')
plt.axis('equal')
plt.tight_layout()
plt.legend()
plt.show()

### Describing the Enrichment Layers

#### Distribution of sanctioned non-individual entities by country

In this section, we introduce two layers of enrichment. The first layer identifies _**Maritime Vessels**_ subject to sanctions and attempts to uncover hidden links to sanctioned entities. Vessels can be used to circumvent trade sanctions and are not always directly linked to sanctioned entities; they are sometimes linked to shell companies that mask the main entity, which may be an organisation, a company or, though this is rare, a high-profile individual. The second layer concerns sanctioned securities. By _**"Sanctioned Security"**_, we mean a tradable (though not necessarily traded) financial asset that holds monetary value. The specialised "Sanctioned Securities" dataset maps sanctioned companies to their tradable financial assets.

##### EL1 Sanctioned vessels

In [ ]:
df_maritime = pd.read_csv(f'{DATA_PATH}/maritime/maritime.csv', low_memory=False) # low_memory=False to avoid dtype warnings
df_maritime['countries'] = df_maritime['countries'].str.upper()
df_maritime['flag'] = df_maritime['flag'].str.upper()

In [ ]:
df_maritime

In [ ]:
country_counts = df_maritime['countries'].value_counts()

country_percentage = country_counts / country_counts.sum()

main_countries = country_counts[country_percentage >= 0.02]
other_countries = country_counts[country_percentage < 0.02].sum()

plot_data = main_countries.copy()
if other_countries > 0:
    plot_data['Others'] = other_countries

fig = plt.figure(figsize=(10, 6))
plot_data.plot(
    kind='pie',
    autopct='%1.1f%%', # autopct to show percentage on the pie chart
    startangle=90,
    pctdistance=0.8
)

plt.title('Distribution of sanctioned vessels by country')
plt.axis('equal')
plt.tight_layout()
plt.legend()
plt.show()

##### EL2 Sanctioned securities

In [ ]:
df_securities = pd.read_csv(f'{DATA_PATH}/securities/securities.csv', low_memory=False)
df_securities['countries'] = df_securities['countries'].str.upper()

In [ ]:
df_securities

The `lei` column contains the Legal Entity Identifier for each security, which is a unique identifier used to identify legal entities participating in financial transactions. The `countries` column indicates the country associated with each security, and the `name` column provides the name of the security. This dataset can be used to analyze the characteristics of sanctioned securities and their associated countries.

In [ ]:
country_counts = df_securities['countries'].value_counts()

country_percentage = country_counts / country_counts.sum()

main_countries = country_counts[country_percentage >= 0.02]
other_countries = country_counts[country_percentage < 0.02].sum()

plot_data = main_countries.copy()
if other_countries > 0:
    plot_data['Others'] = other_countries

fig = plt.figure(figsize=(10, 6))
plot_data.plot(
    kind='pie',
    autopct='%1.1f%%', # autopct to show percentage on the pie chart
    startangle=90,
    pctdistance=0.8
)

plt.title('Distribution of sanctioned individuals by country')
plt.axis('equal')
plt.tight_layout()
plt.legend()
plt.show()

## Part II: Data Analysis and Modeling
---

| # | Question | Use Case | Technology  | Inputs |
| --- | -------- | --------------------- | -------| ------ |
| RQ1 | Which maritime vessels are potentially linked to a sanctioned entity despite not sharing a common ID - i.e. were they renamed or re-registered to conceal ownership? | Sanctions evasion via vessel renaming | **Jaccard Similarity** on character n-grams of `name` field: compare `df_maritime['name']` against `df_consolidated['name']`; pairs above threshold are candidate links | `df_maritime`, `df_consolidated` |
| RQ2 | Is there a statistically significant association between a vessel's country of sanctioned ownership and its registered flag state — in particular, do Russian- and Iranian-linked vessels disproportionately reflag to conduit jurisdictions? | Maritime sanctions evasion; dark fleet detection | Frequent Itemsets (A-Priori): baskets = vessels; items = {country:XX, flag:YY}; derive rule {country:RU} → {flag:PA} and compute confidence and lift against the baseline flag distribution; conduit_jurisdictions variable is the starting point | df_maritime, conduit_jurisdictions |
| RQ3 | Given an entity's current dataset memberships, which additional sanctions programmes is their profile most consistent with — i.e., which programmes have they not yet been listed in but should be? | Cross-list gap detection; targeted investigation | Recommendation System: build a binary entity × dataset utility matrix from datasets column; apply UV decomposition (SGD) to compute latent risk factors; score unobserved (entity, programme) pairs | df_consolidated, df_peps_def |
| RQ4 | Which non-sanctioned PEPs are most similar — by attribute profile — to the entities in df_sanctioned_people_peps, and should therefore be prioritised for enhanced due diligence? | AML/KYC prioritisation | Recommendation System (user-user collaborative filtering): Jaccard similarity on attribute sets between PEP-only entities and sanctioned PEPs; rank PEP-only entities by their average similarity to their nearest sanctioned neighbours | df_peps_def, df_sanctioned_people_peps |

### RQ1: Which maritime vessels are potentially linked to a sanctioned entity despite not sharing a common ID, i.e. were they renamed or re-registered to conceal ownership?

We use Jaccard Similarity to measure lexical similarity on name strings (detecting renamed vessels)

**Jaccard Similarity** on character n-grams of `name` field: compare `df_maritime['name']` against `df_consolidated['name']`; pairs above threshold are candidate links

`df_maritime`, `df_consolidated`

In [ ]:
# Build n-gram sets
def ngrams(s, n=3):
    s = s.lower().strip()
    return set(s[i:i+n] for i in range(len(s) - n + 1))

vessel_names  = df_maritime[['id','caption']].dropna()
entity_names  = df_consolidated[['id','name']].dropna()

vessel_names['ngrams'] = vessel_names['caption'].apply(ngrams)
entity_names['ngrams'] = entity_names['name'].apply(ngrams)

In [ ]:
lsh = MinHashLSH(threshold=0.5, num_perm=64)

entity_hashes = {}
for _, row in entity_names.iterrows():
    m = MinHash(num_perm=64)
    for g in row['ngrams']: m.update(g.encode())
    lsh.insert(row['id'], m)
    entity_hashes[row['id']] = m

candidates = []
for _, row in vessel_names.iterrows():
    m = MinHash(num_perm=64)
    for g in row['ngrams']: m.update(g.encode())
    matches = lsh.query(m)
    for match_id in matches:
        candidates.append({'vessel_id': row['id'], 'vessel_name': row['caption'],
                           'entity_id': match_id})

In [ ]:
entity_ngram_map = entity_names.set_index('id')['ngrams'].to_dict()

results = []
for c in candidates:
    A = entity_ngram_map.get(c['entity_id'], set())
    B = vessel_names.loc[vessel_names['id'] == c['vessel_id'], 'ngrams'].values
    if len(B) == 0: continue
    B = B[0]
    jaccard = len(A & B) / len(A | B) if A | B else 0
    if jaccard >= 0.5:
        results.append({**c, 'entity_name': entity_names.loc[entity_names['id']==c['entity_id'],'name'].values[0],
                        'jaccard': round(jaccard, 3)})

df_rq1 = pd.DataFrame(results).sort_values('jaccard', ascending=False)
df_rq1

# The 0.5 threshold is intentionally loose — vessel names after renaming rarely share more than half their trigrams. Review the output and raise it if there is too much noise.

### RQ2: Is there a statistically significant association between a vessel's country of sanctioned ownership and its registered flag state, in particular, do Russian- and Iranian-linked vessels disproportionately reflag to conduit jurisdictions?

A-Priori on `{country:XX, flag:YY}` baskets.

In [ ]:
conduit_jurisdictions = ['PA', 'VU', 'MT', 'KM', 'PW', 'TV']

In [ ]:
def build_basket(row):
    items = []
    for c in str(row['countries']).split(';'):
        if c.strip(): items.append(f"country:{c.strip()}")
    if pd.notna(row['flag']):
        items.append(f"flag:{row['flag'].strip()}")
        if row['flag'].strip() in conduit_jurisdictions:
            items.append("conduit:True")
    return list(set(items))

df_maritime['basket'] = df_maritime.apply(build_basket, axis=1)

In [ ]:
te = TransactionEncoder()
te_array = te.fit_transform(df_maritime['basket'].tolist())
basket_df = pd.DataFrame(te_array, columns=te.columns_)

freq_items = apriori(basket_df, min_support=0.03, use_colnames=True, max_len=3)

In [ ]:
rules = association_rules(freq_items, metric='confidence', min_threshold=0.4)
rules = rules[rules['lift'] > 1.2].sort_values('lift', ascending=False)

In [ ]:
# Focus on rules that conclude in a conduit flag
flag_rules = rules[rules['consequents'].apply(lambda x: any('flag:' in i or 'conduit:' in i for i in x))]
flag_rules[['antecedents','consequents','support','confidence','lift']]

The `conduit:True` item lets you isolate rules of the form `{country:RU} → {conduit:True}` directly, without having to enumerate each individual flag state. Support at 3% on the maritime dataset should produce a non-trivial rule set — adjust down to 1% if it returns nothing.

### RQ3: Given an entity's current dataset memberships, which additional sanctions programmes is their profile most consistent with i.e., which programmes have they not yet been listed in but should be?

Binary entity × dataset matrix, factorised to find missing listings.

In [ ]:
# Parse the 'datasets' column (semicolon-separated)
df_consolidated['dataset_list'] = df_consolidated['datasets'].str.split(';').apply(
    lambda x: [d.strip() for d in x] if isinstance(x, list) else []
)

all_datasets = sorted({d for lst in df_consolidated['dataset_list'] for d in lst})
entity_ids   = df_consolidated['id'].tolist()

dataset_idx  = {d: i for i, d in enumerate(all_datasets)}
entity_idx   = {e: i for i, e in enumerate(entity_ids)}

M = np.zeros((len(entity_ids), len(all_datasets)), dtype=np.float32)
for _, row in df_consolidated.iterrows():
    i = entity_idx[row['id']]
    for d in row['dataset_list']:
        if d in dataset_idx:
            M[i, dataset_idx[d]] = 1.0

In [ ]:
np.random.seed(42)
k   = 10   # latent dimensions — keep small for speed
eta = 0.01
lam = 0.02

U = np.random.normal(0, 0.1, (len(entity_ids), k)).astype(np.float32)
V = np.random.normal(0, 0.1, (k, len(all_datasets))).astype(np.float32)

rows, cols = np.nonzero(M)
for epoch in range(5):
    perm = np.random.permutation(len(rows))
    for p in perm:
        i, j  = rows[p], cols[p]
        e     = M[i,j] - U[i] @ V[:,j]
        U[i] += eta * (e * V[:,j] - lam * U[i])
        V[:,j]+= eta * (e * U[i]  - lam * V[:,j])
    print(f"Epoch {epoch+1} done")

In [ ]:
scores = U @ V   # predicted relevance for every (entity, dataset) pair

# Mask already-known entries so we only look at gaps
scores[M == 1] = -np.inf

# Top-5 missing dataset recommendations per entity (sample: first 20 entities)
for i in range(20):
    top_datasets = np.argsort(scores[i])[::-1][:5]
    entity_name  = df_consolidated.iloc[i]['name']
    recs         = [all_datasets[j] for j in top_datasets]
    print(f"{entity_name}: {recs}")

### RQ4: Which non-sanctioned PEPs are most similar, by attribute profile, to the entities in df_sanctioned_people_peps, and should therefore be prioritised for enhanced due diligence?

Jaccard on attribute sets to rank unsanctioned PEPs by similarity to sanctioned ones.

In [ ]:
def attr_set(row, country_col='countries', dataset_col='datasets'):
    countries = {f"c:{x.strip()}" for x in str(row.get(country_col,'')).split(';') if x.strip()}
    datasets  = {f"d:{x.strip()}" for x in str(row.get(dataset_col,'')).split(';') if x.strip()}
    return countries | datasets

# Sanctioned PEPs (positive class)
sanctioned_sets = df_sanctioned_people_peps.apply(attr_set, axis=1).tolist()

# Non-sanctioned PEPs (to score)
non_sanctioned  = df_peps_def[~df_peps_def['id'].isin(df_sanctioned_people_peps['id'])].copy()
non_sanctioned['attr_set'] = non_sanctioned.apply(attr_set, axis=1)

In [ ]:
def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 0.0

# Mean Jaccard to the nearest 50 sanctioned PEPs (sample for speed)
sample_sanctioned = sanctioned_sets[:50]

non_sanctioned['risk_score'] = non_sanctioned['attr_set'].apply(
    lambda s: np.mean([jaccard(s, ref) for ref in sample_sanctioned])
)

In [ ]:
non_sanctioned.sort_values('risk_score', ascending=False)[['name','countries','risk_score']].head(20)

The sample of 50 sanctioned PEPs is a speed concession — computing against all 3,000+ sanctioned PEPs would be slow without vectorisation. If the results look thin, increase the sample. You can also replace the mean with the max similarity to favour PEPs that are very close to at least one sanctioned entity, rather than broadly similar to many.

## Part III: Machine Learning and Predictive Modeling
---

### Can we predict whether a PEP will also appear in the consolidated sanctions list using only the features available in their PEP profile (country count, alias count, identifier count, dataset count, top-country indicators)?

Label definition is clear: `1` if entity ID is in `df_sanctioned_people_peps`, `0` otherwise. 

Leave the heading where it is and fill it under the Part III cell.

### How much better does a high-dimensional SGD classifier perform over LDA/QDA when one-hot country features are added, and what does the gap reveal about the linearity assumption?

Model benchmarking; methodology justification

Large-Scale ML (SGD classifier): same binary task as RQ5 but with full one-hot country features; compare AUC/AP against LDA/QDA baseline; the performance gap characterises whether risk is linearly separable in geographic feature space

Same inputs as of RQ5, plus one-hot encoding of `countries` 